# Tutorial

In [1]:
import pandas as pd
import pint
ureg = pint.get_application_registry()

In [2]:
from aircraftdetective.processing.usdot import process_data_usdot_t2
from aircraftdetective.processing.acftdb import _read_engine_database
from aircraftdetective.processing.literature import (
    process_data_weinold_database,
    process_data_babikian_figures
)

from aircraftdetective.calculations.engines import (
    determine_takeoff_to_cruise_tsfc_ratio,
    scale_engine_data_from_icao_emissions_database
)
from aircraftdetective.calculations.engines import (
    calculate_air_mass_flow_rate,
    calculate_engine_efficiencies
)
from aircraftdetective.calculations.aerodynamics import (
    compute_lift_to_drag_ratio,
    compute_aspect_ratio
)

from aircraftdetective.utility.tabular import (
    left_merge_wildcard,
    export_typed_dataframe_to_excel
)

Aircraft specifications are provided in an Excel file hosted at the Zenodo repository ["Aircraft Detective" Dataset](https://doi.org/10.5281/zenodo.14382100).  
They can be loaded using the [`aircraftdetective.processing.literature.process_data_weinold_database`](https://aircraftdetective.readthedocs.io/en/latest/api/processing/literature/#aircraftdetective.processing.literature.process_data_weinold_database) function.

In [5]:
df_acft = process_data_weinold_database()

In [6]:
df_acft.head(5)

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,MZFW,Number of Engines,Sources (Weights),Comments,Payload/Range: Range at Point B,Payload/Range: Range at Point C,Payload/Range: MZFW at Point B,Payload/Range: MZFW at Point C,Payload/Range: MTOW,Source (Payload/Range)
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,50349.0,2.0,https://aircraft.airbus.com/sites/g/files/jlcb...,NaN,3598.3749284019223,7158.606627935522,50350.0026357575,43169.9977927272,60781.0,https://aircraft.airbus.com/sites/g/files/jlcb...
1,Airbus,A220-300,NaN,A220-300,A220-300 BD-500-1A11,Narrow,2016,PW1521G,PW1521G,6300.0,...,55792.0,2.0,https://aircraft.airbus.com/sites/g/files/jlcb...,NaN,3681.3845400518944,6230.448593959429,55791.8615,49986.0186311349,67585.0,https://aircraft.airbus.com/sites/g/files/jlcb...
2,Airbus,A300-600,A300-600,A300 B4-600,Airbus Industrie A300-600/R/CF/RCF,Wide,1984,CF6-80C2A5,CF6-80C2A5,6852.0,...,130000.0,2.0,[JAWA:07/08],NaN,3621.6888888888893,5700.044444444445,129255.84717830301,115645.08717830303,165035.00191999998,https://www.airbus.com/sites/g/files/jlcbta136...
3,Airbus,A300-600,A300-600,A300 B4-601,Airbus Industrie A300-600/R/CF/RCF,Wide,1986,PW4158,PW4158,6852.0,...,130000.0,2.0,[JAWA:07/08],NaN,3621.6888888888893,5700.044444444445,129255.84717830301,115645.08717830303,165035.00191999998,https://www.airbus.com/sites/g/files/jlcbta136...
4,Airbus,A310-200,NaN,A310-200,Airbus Industrie A310-200C/F,Wide,1983,JT9D-7R4D*,JT9D-7R4D*,6500.0,...,nan,2.0,[JAWA:07/08],NaN,nan,nan,nan,nan,nan,https://www.airbus.com/sites/g/files/jlcbta136...


Engine specifications are provided in an Excel file hosted at the Zenodo repository ["Aircraft Detective" Dataset](https://doi.org/10.5281/zenodo.14382100).

In [7]:
dict_tsfc_scaling = determine_takeoff_to_cruise_tsfc_ratio(degree=2, plot=True)
df_engines_scaled = scale_engine_data_from_icao_emissions_database(
    scaling_polynomial=dict_tsfc_scaling['TSFC (cruise)'],
)
df_engines_scaled.drop(columns=['Final Test Date'], inplace=True)

In [8]:
df_engines = _read_engine_database()

In [9]:
df_merged = left_merge_wildcard(
    df_left=df_acft,
    df_right=df_engines_scaled,
    left_on='Engine Designation (ICAO)',
    right_on='Engine Identification',
)
df_merged = left_merge_wildcard(
    df_left=df_merged,
    df_right=df_engines,
    left_on='Engine Designation (aircraft-database.com)',
    right_on='Engine Designation',
)

In [10]:
df_merged.head(4)

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,TSFC (takeoff),TSFC (cruise),Bypass Ratio,Overall Pressure Ratio,Dry Weight,Fan Diameter,Max. Continuous Thrust,_id_engine,Engine Family,Engine Manufacturer
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,7.048658481127786,13.44781176238713,nan,nan,2177.0,1.85,83.12,1ec8d740-0103-60f2-860d-69edf93e810e,turbofan,Pratt & Whitney
1,Airbus,A220-300,NaN,A220-300,A220-300 BD-500-1A11,Narrow,2016,PW1521G,PW1521G,6300.0,...,7.060990585345886,13.465996936432283,nan,nan,2177.0,1.85,92.35,1ec8d742-7536-6fda-860d-9318256fabb1,turbofan,Pratt & Whitney
2,Airbus,A300-600,A300-600,A300 B4-600,Airbus Industrie A300-600/R/CF/RCF,Wide,1984,CF6-80C2A5,CF6-80C2A5,6852.0,...,9.654088834576154,16.912337735297225,5.1,nan,4300.0,2.36,250.03,1ec89968-74fb-670e-b99a-cf6db5bd217d,turbofan,General Electric
3,Airbus,A300-600,A300-600,A300 B4-601,Airbus Industrie A300-600/R/CF/RCF,Wide,1986,PW4158,PW4158,6852.0,...,9.616279069767442,16.867485204335395,4.8,30.7,4273.0,2.39,220.54,1ec8c156-ca49-61aa-860d-a11b73be09ea,turbofan,Pratt & Whitney


In [11]:
df_merged = calculate_air_mass_flow_rate(df_merged)

In [12]:
df_t2 = process_data_usdot_t2()
df_t2 = df_t2.groupby('Aircraft Designation (US DOT Schedule T2)').mean()
df_with_dot = pd.merge(
    how='left',
    left=df_merged,
    right=df_t2,
    left_on='Aircraft Designation (US DOT Schedule T2)',
    right_on='Aircraft Designation (US DOT Schedule T2)'
)

In [13]:
df_with_dot = calculate_engine_efficiencies(df=df_with_dot)
df_with_dot = compute_aspect_ratio(df_with_dot)
df_with_dot = compute_lift_to_drag_ratio(
    df_with_dot,
    beta=
    {
        'Narrow': 0.06,
        'Wide': 0.04,
        'Regional': 0.12,
    })

In [14]:
df_with_dot['Energy Use (per ASK)'] = df_with_dot['Energy Use (per ASK)'].pint.to('MJ/km')

In [16]:
from aircraftdetective.calculations.weight import calculate_weight_metrics

In [17]:
df_with_dot = calculate_weight_metrics(df_with_dot)

In [18]:
from aircraftdetective.utility.tabular import update_column_data

In [85]:
df_literature = process_data_weinold_database(sheet_name='Literature Data')

In [86]:
df_literature.dtypes

Aircraft Designation (Literature)                                       object
L/D                                               pint[dimensionless][float64]
Source (L/D)                                                            object
Energy Use (per ASK)                      pint[megajoule / kilometer][float64]
Source (Energy Use)                                                     object
TSFC (cruise)                        pint[gram / kilonewton / second][float64]
Source (TSFC (Cruise))                                                  object
OEW/MTOW                                          pint[dimensionless][float64]
Source (OEW/MTOW)                                                       object
dtype: object

In [87]:
df_literature

,Aircraft Designation (Literature),L/D,Source (L/D),Energy Use (per ASK),Source (Energy Use),TSFC (cruise),Source (TSFC (Cruise)),OEW/MTOW,Source (OEW/MTOW)
0,A300-600,14.6026,https://doi.org/10.5281/zenodo.14560914,1.3176,https://doi.org/10.5281/zenodo.14560914,16.8925,https://doi.org/10.5281/zenodo.14560914,0.52078,https://doi.org/10.5281/zenodo.14560914
1,A310-300,16.499,https://doi.org/10.5281/zenodo.14560914,1.4459,https://doi.org/10.5281/zenodo.14560914,16.8925,https://doi.org/10.5281/zenodo.14560914,0.50967,https://doi.org/10.5281/zenodo.14560914
2,A320-100/200,15.9818,https://doi.org/10.5281/zenodo.14560914,1.152,https://doi.org/10.5281/zenodo.14560914,16.8925,https://doi.org/10.5281/zenodo.14560914,0.55411,https://doi.org/10.5281/zenodo.14560914
3,A380-800,nan,NaN,1.375286583998318,Rohrer et al.,nan,NaN,nan,NaN
4,B707-300,15.8095,https://doi.org/10.5281/zenodo.14560914,3.0372,https://doi.org/10.5281/zenodo.14560914,28.1765,https://doi.org/10.5281/zenodo.14560914,28.1765,https://doi.org/10.5281/zenodo.14560914
5,B720-000,15.8095,https://doi.org/10.5281/zenodo.14560914,3.0372,https://doi.org/10.5281/zenodo.14560914,nan,NaN,27.0278,https://doi.org/10.5281/zenodo.14560914
6,B720-000,nan,NaN,3.2365,https://doi.org/10.5281/zenodo.14560914,nan,NaN,nan,NaN
7,B727-200/231A,13.798,https://doi.org/10.5281/zenodo.14560914,2.0912,https://doi.org/10.5281/zenodo.14560914,23.244,https://doi.org/10.5281/zenodo.14560914,0.57158,https://doi.org/10.5281/zenodo.14560914
8,B737-100/200,13.9129,https://doi.org/10.5281/zenodo.14560914,1.9257,https://doi.org/10.5281/zenodo.14560914,22.771,https://doi.org/10.5281/zenodo.14560914,0.51984,https://doi.org/10.5281/zenodo.14560914
9,B737-300,14.0854,https://doi.org/10.5281/zenodo.14560914,1.3885,https://doi.org/10.5281/zenodo.14560914,20.8791,https://doi.org/10.5281/zenodo.14560914,0.54777,https://doi.org/10.5281/zenodo.14560914


In [100]:
dftest = update_column_data(
    df_main=df_with_dot,
    df_other=df_literature,
    merge_column='Aircraft Designation (Literature)',
    list_columns=['L/D', 'Energy Use (per ASK)', 'TSFC (cruise)', 'OEW/MTOW']
)

In [19]:
df_with_dot.columns

Index(['Manufacturer', 'Aircraft Designation',
       'Aircraft Designation (Literature)',
       'Aircraft Designation (aircraft-database.com)',
       'Aircraft Designation (US DOT Schedule T2)', 'Type', 'YOI',
       'Engine Designation (ICAO)',
       'Engine Designation (aircraft-database.com)', 'Design Range',
       'Sources (Design Range)', 'Cruise Speed', 'Sources (Cruise Speed)',
       'Wingspan', 'Wing Area', 'Sources (Wing Dimensions)', 'Pax Exit Limit',
       'Average Pax', 'Sources (Pax)', 'OEW', 'MTOW', 'MZFW',
       'Number of Engines', 'Sources (Weights)', 'Comments',
       'Payload/Range: Range at Point B', 'Payload/Range: Range at Point C',
       'Payload/Range: MZFW at Point B', 'Payload/Range: MZFW at Point C',
       'Payload/Range: MTOW', 'Source (Payload/Range)', 'Fuel Flow (takeoff)',
       'Fuel Flow (climbout)', 'Fuel Flow (approach)', 'Fuel Flow (idle)',
       'B/P Ratio', 'Pressure Ratio', 'Rated Thrust', 'TSFC (takeoff)',
       'TSFC (cruise)', 'By

In [20]:
from aircraftdetective.calculations.decomposition import compute_efficiency_improvement_metrics
from aircraftdetective.utility.statistics import _compute_polynomials_from_dataframe

In [21]:
_compute_polynomials_from_dataframe(
    df=df_with_dot,
    col_name_x='YOI',
    list_col_names_y=['Energy Use (per ASK)'],
    degree=2,
    plot=True
)

{'Energy Use (per ASK)': Polynomial([ 1.56486511, -0.40996086, -0.11409516], domain=[1964., 2018.], window=[-1.,  1.], symbol='x'),
 'Energy Use (per ASK)_r2': np.float64(0.19444020819090868)}

In [22]:
_compute_polynomials_from_dataframe(
    df=df_with_dot,
    col_name_x='YOI',
    list_col_names_y=['L/D'],
    degree=2,
    plot=True
)

{'L/D': Polynomial([ 23.71012949,   2.40815381, -10.74860272], domain=[1958., 2018.], window=[-1.,  1.], symbol='x'),
 'L/D_r2': np.float64(0.07515808307512606)}

In [23]:
_compute_polynomials_from_dataframe(
    df=df_with_dot,
    col_name_x='YOI',
    list_col_names_y=['TSFC (cruise)'],
    degree=2,
    plot=True
)

{'TSFC (cruise)': Polynomial([ 1.83114287e+01, -3.94726162e+00, -4.71920340e-03], domain=[1958., 2018.], window=[-1.,  1.], symbol='x'),
 'TSFC (cruise)_r2': np.float64(0.6759842850610415)}

In [25]:
_compute_polynomials_from_dataframe(
    df=df_with_dot,
    col_name_x='YOI',
    list_col_names_y=['norm(OEW/Exit Limit)'],
    degree=2,
    plot=True
)

{'norm(OEW/Exit Limit)': Polynomial([ 0.65281976,  0.01910429, -0.07476659], domain=[1952., 2018.], window=[-1.,  1.], symbol='x'),
 'norm(OEW/Exit Limit)_r2': np.float64(0.006883739296970615)}